In [1]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "findspark"])

0

In [2]:
from pyspark.sql import SparkSession

In [3]:
mysql_jar = "/home/jovyan/work/jars/mysql-connector-j-8.3.0.jar"
cassandra_jar = "/home/jovyan/work/jars/spark-cassandra-connector-assembly_2.12-3.5.0.jar"

In [4]:
spark = SparkSession.builder \
    .appName("Test_Full_Connectivity") \
    .config("spark.jars", f"{mysql_jar},{cassandra_jar}") \
    .config("spark.cassandra.connection.host", "cassandra") \
    .getOrCreate()

In [5]:
df = spark.read \
        .format("jdbc") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .option("url", "jdbc:mysql://mysql:3306/etl_db") \
        .option("dbtable", "application") \
        .option("user", "root") \
        .option("password", "1") \
        .load()
print(">> MySQL: OK!")
df.show(5)

>> MySQL: OK!
+---+--------------------+--------------------+-------------------+--------------------+---------+----------+---------+------------+------+-----------+------+--------------------+---------+--------+------------+-----------+-----+-------------------+------+-------+---------------+-------+---------------------+-------------------+------------+
| id|          created_by|        created_date|   last_modified_by|  last_modified_date|is_active|first_name|last_name|phone_number|  city|postal_code|status|              cv_url|is_review|is_match|is_contacted|is_rejected|score|              email|job_id|user_id|conversation_id|country|is_notification_email|is_notification_sms|publisher_id|
+---+--------------------+--------------------+-------------------+--------------------+---------+----------+---------+------------+------+-----------+------+--------------------+---------+--------+------------+-----------+-----+-------------------+------+-------+---------------+-------+----------

In [6]:
df_cassandra = spark.read \
        .format("org.apache.spark.sql.cassandra") \
        .options(table='tracking', keyspace='recruitment') \
        .load()
print(">> Cassandra: OK!")
df_cassandra.show(5)

>> Cassandra: OK!
+--------------------+----+----------+-----------+---+------------+-----+--------------------+---------------+--------------------+---+--------+----+------+----+------------+----+---------+--------------------+----+--------------------+-------------------+------------+-----------+----------+----------+--------+---+--------+
|         create_time| bid|        bn|campaign_id| cd|custom_track|   de|                  dl|             dt|                  ed| ev|group_id|  id|job_id|  md|publisher_id|  rl|       sr|                  ts|  tz|                  ua|                uid|utm_campaign|utm_content|utm_medium|utm_source|utm_term|  v|      vp|
+--------------------+----+----------+-----------+---+------------+-----+--------------------+---------------+--------------------+---+--------+----+------+----+------------+----+---------+--------------------+----+--------------------+-------------------+------------+-----------+----------+----------+--------+---+--------+
|1ee